In [ ]:
# ============================================================
# AWS GLUE E-COMMERCE ETL
# Historical CSV + Ordering API JSON
# S3 Curated + Amazon Redshift
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce

In [ ]:
# ============================================================
# 1. SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("Ecommerce-Historical-API-ETL") \
    .getOrCreate()

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BUCKET = "your-bucket-name"

# Historical data
RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"

# API incremental data
API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders"

# Curated data
PROCESSED_PATH = f"s3://{BUCKET}/curated/ecommerce"

In [ ]:
# # ============================================================
# # REDSHIFT CONFIGURATION
# # ============================================================

# REDSHIFT_CONNECTION = "ecommerce-redshift-connection"

# REDSHIFT_DATABASE = "dev"

# REDSHIFT_SCHEMA = "public"

# # S3 temporary directory used by Spark/Glue
# REDSHIFT_TEMP_DIR = (
#     f"s3://{BUCKET}/redshift-temp/"
# )

In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_column_names(df):

    for column in df.columns:

        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(
            column,
            new_name
        )

    return df


def read_csv(filename):

    path = f"{RAW_PATH}/{filename}"

    print("Reading:", path)

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .csv(path)
    )


def write_parquet(df, table_name):

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )

    print(
        f"Writing {table_name} -> {output_path}"
    )

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

In [ ]:
# ============================================================
# 4. READ HISTORICAL DATA
# ============================================================

print("============================================")
print("READING HISTORICAL DATA")
print("============================================")


customers = read_csv("customers.csv")

categories = read_csv("categories.csv")

products = read_csv("products.csv")

departments = read_csv("departments.csv")

employees = read_csv("employees.csv")

suppliers = read_csv("suppliers.csv")

orders = read_csv("orders.csv")

order_details = read_csv("order_details.csv")

payments = read_csv("payments.csv")

product_suppliers = read_csv(
    "product_suppliers.csv"
)

shippers = read_csv("shippers.csv")

shipments = read_csv("shipments.csv")

In [ ]:
customers.printSchema()                  
categories.printSchema()
products.printSchema()
departments.printSchema()
employees.printSchema()
suppliers.printSchema()
orders.printSchema()
order_details.printSchema()
payments.printSchema()
product_suppliers.printSchema()
shippers.printSchema()
shipments.printSchema()

In [ ]:
# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

print("============================================")
print("CLEANING COLUMN NAMES")
print("============================================")


customers = clean_column_names(customers)

categories = clean_column_names(categories)

products = clean_column_names(products)

departments = clean_column_names(departments)

employees = clean_column_names(employees)

suppliers = clean_column_names(suppliers)

orders = clean_column_names(orders)

order_details = clean_column_names(order_details)

payments = clean_column_names(payments)

product_suppliers = clean_column_names(
    product_suppliers
)

shippers = clean_column_names(shippers)

shipments = clean_column_names(shipments)

In [ ]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

In [ ]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="your-bucket-name",
    Prefix="api_raw/orders/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

In [ ]:
from pyspark.sql import functions as F

BUCKET = "your-bucket-name"

API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders/"

api_orders = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API records:", api_orders.count())

api_orders.printSchema()

In [ ]:
api_orders_df = api_orders.select(
    
    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("order.CustomerID")
        .cast("long")
        .alias("customerid"),

    F.to_date(
        F.col("order.OrderDate")
    ).alias("orderdate"),

    F.upper(
        F.trim(
            F.col("order.Status")
        )
    ).alias("status"),

    # API metadata
    F.col("metadata.event_id")
        .alias("event_id"),

    F.to_timestamp(
        F.col("metadata.event_timestamp")
    ).alias("event_timestamp"),

    F.col("metadata.event_type")
        .alias("event_type"),

    F.col("metadata.source")
        .alias("source")
)

api_orders_df.show(10, truncate=False)

In [ ]:
api_order_details_df = (
    api_orders
    .select(
        
        F.col("order.OrderID")
            .cast("long")
            .alias("orderid"),

        F.explode(
            F.col("order_details")
        ).alias("detail")
    )
)
api_order_details_df.show()

In [ ]:
api_order_details_df = api_order_details_df.select(

    "orderid",

    F.col("detail.OrderDetailID")
        .cast("long")
        .alias("orderdetailid"),

    F.col("detail.ProductID")
        .cast("long")
        .alias("productid"),

    F.col("detail.Quantity")
        .cast("long")
        .alias("quantity"),

    F.col("detail.UnitPrice")
        .cast("decimal(12,2)")
        .alias("unitprice"),

    F.col("detail.Discount")
        .cast("decimal(12,2)")
        .alias("discount")
)
api_order_details_df.show()

In [ ]:
api_payments_df = api_orders.select(

    F.col("payment.PaymentID")
        .cast("long")
        .alias("paymentid"),

    F.col("payment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.trim(
        F.col("payment.PaymentMethod")
    ).alias("paymentmethod"),

    F.to_date(
        F.col("payment.PaymentDate")
    ).alias("paymentdate"),

    F.col("payment.Amount")
        .cast("decimal(12,2)")
        .alias("amount"),

    F.upper(
        F.trim(
            F.col("payment.PaymentStatus")
        )
    ).alias("paymentstatus")
)

api_payments_df.show(10)

In [ ]:
api_shipments_df = api_orders.select(

    F.col("shipment.ShipmentID")
        .cast("long")
        .alias("shipmentid"),

    F.col("shipment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("shipment.ShipperID")
        .cast("long")
        .alias("shipperid"),

    F.to_date(
        F.col("shipment.ShipDate")
    ).alias("shipdate"),

    F.to_date(
        F.col("shipment.DeliveryDate")
    ).alias("deliverydate")
)

api_shipments_df.show(10)

In [ ]:
api_order_summary_df = api_orders.select(

    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("summary.subtotal")
        .cast("decimal(12,2)")
        .alias("subtotal"),

    F.col("summary.discount")
        .cast("decimal(12,2)")
        .alias("discount"),

    F.col("summary.tax_rate")
        .cast("decimal(5,2)")
        .alias("tax_rate"),

    F.col("summary.tax_amount")
        .cast("decimal(12,2)")
        .alias("tax_amount"),

    F.col("summary.shipping_cost")
        .cast("decimal(12,2)")
        .alias("shipping_cost"),

    F.col("summary.total_amount")
        .cast("decimal(12,2)")
        .alias("total_amount")
)
api_order_summary_df.show(10)

In [ ]:
historical_orders = orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [ ]:
new_orders = api_orders_df.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [ ]:
all_orders = (
    historical_orders
    .unionByName(new_orders)
    .dropDuplicates(["orderid"])
)
all_orders.show(10)

In [ ]:
print("Historical orders:", historical_orders.count())
print("API orders:", new_orders.count())
print("Combined orders:", all_orders.count())

In [ ]:
historical_orders_details = order_details.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [ ]:
new_orders_details = api_order_details_df.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [ ]:
all_orders_details = (
    historical_orders_details
    .unionByName(new_orders_details)
    .dropDuplicates(["OrderDetailID"])
)
all_orders_details.show(10)

In [ ]:
print("Historical orders_details:", historical_orders_details.count())
print("API orders_details:", new_orders_details.count())
print("Combined orders_details:", all_orders_details.count())

In [ ]:
historical_payments = payments.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [ ]:
new_orders_payments = api_payments_df.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [ ]:
all_payments = (
    historical_payments
    .unionByName(new_orders_payments)
    .dropDuplicates(["paymentid"])
)
all_payments.show(10)

In [ ]:
print("Historical payments:", historical_payments.count())
print("API payments:", new_orders_payments.count())
print("Combined payments:", all_payments.count())

In [ ]:
historical_shipments = shipments.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [ ]:
new_orders_shipments = api_shipments_df.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [ ]:
all_shipments = (
    historical_shipments
    .unionByName(new_orders_shipments)
    .dropDuplicates(["ShipmentID"])
)
all_shipments.show(10)

In [ ]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

In [ ]:
all_shipments.groupBy("ShipmentID") \
    .count() \
    .filter("count > 1") \
    .show()

In [ ]:
all_shipments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in all_shipments.columns
]).show()

In [ ]:
all_shipments = all_shipments.dropna(how="all")

In [ ]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

In [ ]:
all_orders_details = (
    all_orders_details

    # Make sure types are consistent
    .withColumn("orderdetailid", F.col("orderdetailid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("long"))
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

    # Handle negative values
    .withColumn(
        "quantity",
        F.when(F.col("quantity") < 0, 0)
         .otherwise(F.col("quantity"))
    )

    .withColumn(
        "unitprice",
        F.when(F.col("unitprice") < 0, 0)
         .otherwise(F.col("unitprice"))
    )

    .withColumn(
        "discount",
        F.when(F.col("discount") < 0, 0)
         .otherwise(F.col("discount"))
    )

    # Recalculate amounts
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unitprice")
    )

    .withColumn(
        "discount_amount",
        F.col("gross_amount") * F.col("discount") / 100
    )

    .withColumn(
        "total_amount",
        F.col("gross_amount") - F.col("discount_amount")
    )
)

In [ ]:
all_orders_details.show(2)

In [ ]:
all_orders_details.count()

In [ ]:
all_api_orders = (
    api_orders_df
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
)
all_api_orders.show(10)

In [ ]:
API_CURATED_PATH = (
    f"s3://{BUCKET}/curated/ecommerce/api_orders"
)

(
    all_api_orders
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .save(API_CURATED_PATH)
)

In [ ]:
dim_customer = (
    customers
    .select(
        F.col("customerid").cast("long").alias("customer_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.lower(F.trim("email")).alias("email"),
        F.trim("city").alias("city"),
        F.trim("country").alias("country"),
        F.to_date("registrationdate").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
)

In [ ]:
dim_category = (
    categories
    .select(
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("categoryname").alias("category_name")
    )
    .dropDuplicates(["category_id"])
)

In [ ]:
dim_department = (
    departments
    .select(
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("departmentname").alias("department_name")
    )
    .dropDuplicates(["department_id"])
)

In [ ]:
dim_product = (
    products
    .select(
        F.col("productid").cast("long").alias("product_id"),
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("productname").alias("product_name"),
        F.trim("brand").alias("brand"),
        F.col("price").cast("decimal(12,2)").alias("price"),
        F.col("cost").cast("decimal(12,2)").alias("cost"),
        F.col("stock").cast("int").alias("stock")
    )
    .dropDuplicates(["product_id"])
)

In [ ]:
dim_supplier = (
    suppliers
    .select(
        F.col("supplierid").cast("long").alias("supplier_id"),
        F.trim("suppliername").alias("supplier_name"),
        F.trim("country").alias("country")
    )
    .dropDuplicates(["supplier_id"])
)

In [ ]:
dim_employee = (
    employees
    .select(
        F.col("employeeid").cast("long").alias("employee_id"),
        F.col("managerid").cast("long").alias("manager_id"),
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.col("salary").cast("decimal(12,2)").alias("salary"),
        F.to_date("hiredate").alias("hire_date")
    )
    .dropDuplicates(["employee_id"])
)

In [ ]:
dim_shipper = (
    shippers
    .select(
        F.col("shipperid").cast("long").alias("shipper_id"),
        F.trim("companyname").alias("company_name")
    )
    .dropDuplicates(["shipper_id"])
)

In [ ]:
date_range = (
    all_orders
    .select(
        F.to_date("orderdate").alias("date")
    )
    .filter(F.col("date").isNotNull())
    .agg(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

In [ ]:
dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("interval 1 day")
            )
        ).alias("date")
    )
)

In [ ]:
dim_date = (
    dim_date
    .withColumn(
        "date_id",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )
    .withColumn(
        "day",
        F.dayofmonth("date")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )
    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )
)

In [ ]:
payment_methods = (
    all_payments
    .select(
        F.upper(
            F.trim("paymentmethod")
        ).alias("payment_method")
    )
    .filter(
        F.col("payment_method").isNotNull()
    )
    .distinct()
)

In [ ]:
window_payment = Window.orderBy("payment_method")

dim_payment_method = (
    payment_methods
    .withColumn(
        "payment_method_id",
        F.row_number().over(window_payment)
    )
    .select(
        "payment_method_id",
        "payment_method"
    )
)

In [ ]:
order_statuses = (
    all_orders
    .select(
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
    .filter(
        F.col("order_status").isNotNull()
    )
    .distinct()
)

In [ ]:
window_status = Window.orderBy("order_status")

dim_order_status = (
    order_statuses
    .withColumn(
        "order_status_id",
        F.row_number().over(window_status)
    )
    .select(
        "order_status_id",
        "order_status"
    )
)

In [ ]:
fact_order = (
    all_orders
    .select(
        F.col("orderid").cast("long").alias("order_id"),
        F.col("customerid").cast("long").alias("customer_id"),
        F.to_date("orderdate").alias("order_date"),
        F.date_format(
            F.to_date("orderdate"),
            "yyyyMMdd"
        ).cast("int").alias("date_id"),
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
)

In [ ]:
fact_order = (
    fact_order
    .join(
        dim_order_status,
        fact_order.order_status ==
        dim_order_status.order_status,
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "date_id",
        "order_date",
        "order_status_id"
    )
)

In [ ]:
fact_order_detail = (
    all_orders_details
    .select(
        F.col("orderdetailid")
            .cast("long")
            .alias("order_detail_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("quantity")
            .cast("int")
            .alias("quantity"),

        F.col("unitprice")
            .cast("decimal(12,2)")
            .alias("unit_price"),

        F.col("discount")
            .cast("decimal(12,2)")
            .alias("discount"),

        F.col("gross_amount")
            .cast("decimal(14,2)")
            .alias("gross_amount"),

        F.col("discount_amount")
            .cast("decimal(14,2)")
            .alias("discount_amount"),

        F.col("total_amount")
            .cast("decimal(14,2)")
            .alias("total_amount")
    )
    .dropDuplicates(["order_detail_id"])
)

In [ ]:
fact_order_detail.show(5)

In [ ]:
fact_payment_base = (
    all_payments
    .select(
        F.col("paymentid")
            .cast("long")
            .alias("payment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.to_date("paymentdate")
            .alias("payment_date"),

        F.date_format(
            F.to_date("paymentdate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("date_id"),

        F.upper(
            F.trim("paymentmethod")
        )
        .alias("payment_method"),

        F.col("amount")
            .cast("decimal(14,2)")
            .alias("amount")
    )
)

In [ ]:
fact_payment = (
    fact_payment_base
    .join(
        dim_payment_method,
        fact_payment_base.payment_method ==
        dim_payment_method.payment_method,
        "left"
    )
    .select(
        "payment_id",
        "order_id",
        "date_id",
        "payment_date",
        "payment_method_id",
        "amount"
    )
)

In [ ]:
fact_shipment = (
    all_shipments
    .select(
        F.col("shipmentid")
            .cast("long")
            .alias("shipment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("shipperid")
            .cast("long")
            .alias("shipper_id"),

        F.to_date("shipdate")
            .alias("ship_date"),

        F.to_date("deliverydate")
            .alias("delivery_date")
    )
    .withColumn(
        "shipping_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
)

In [ ]:
fact_shipment.show(2)

In [ ]:
customer_sales_base = (
    fact_order_detail
    .join(
        fact_order.select(
            "order_id",
            "customer_id",
            "order_date"
        ),
        "order_id",
        "inner"
    )
)

In [ ]:
fact_customer_sales = (
    customer_sales_base
    .groupBy("customer_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "total_amount"
        ).alias("average_order_line"),

        F.min(
            "order_date"
        ).alias("first_order_date"),

        F.max(
            "order_date"
        ).alias("last_order_date")
    )
)

In [ ]:
fact_customer_sales = (
    fact_customer_sales
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_sales") >= 10000,
            "VIP"
        )
        .when(
            F.col("total_sales") >= 5000,
            "PREMIUM"
        )
        .when(
            F.col("total_sales") >= 1000,
            "REGULAR"
        )
        .otherwise(
            "LOW_VALUE"
        )
    )
)

In [ ]:
fact_customer_sales.show(1)

In [ ]:
fact_product_sales = (
    fact_order_detail
    .groupBy("product_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("units_sold"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "unit_price"
        ).alias("average_unit_price")
    )
)

In [ ]:
bridge_product_supplier = (
    product_suppliers
    .select(
        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("supplierid")
            .cast("long")
            .alias("supplier_id")
    )
    .dropDuplicates(
        ["product_id", "supplier_id"]
    )
)

In [ ]:
write_parquet(dim_customer, "dim_customer")
write_parquet(dim_product, "dim_product")
write_parquet(dim_category, "dim_category")
write_parquet(dim_department, "dim_department")
write_parquet(dim_supplier, "dim_supplier")
write_parquet(dim_employee, "dim_employee")
write_parquet(dim_shipper, "dim_shipper")
write_parquet(dim_date, "dim_date")
write_parquet(dim_payment_method, "dim_payment_method")
write_parquet(dim_order_status, "dim_order_status")

In [ ]:
write_parquet(fact_order, "fact_order")
write_parquet(fact_order_detail, "fact_order_detail")
write_parquet(fact_payment, "fact_payment")
write_parquet(fact_shipment, "fact_shipment")
write_parquet(fact_customer_sales, "fact_customer_sales")
write_parquet(fact_product_sales, "fact_product_sales")

In [ ]:
write_parquet(bridge_product_supplier,"bridge_product_supplier")